In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

print('Imports OK')

In [ ]:
# ============================================================
# 1-A  Parse pixel array (從任何格式解析成 1D float array)
# ============================================================
def parse_pixel_array(val):
    """
    把儲存格內容統一轉成 1D numpy array(float)。
    支援: None / scalar / list / np.ndarray / string
    0 值視為無效像素（雲、遮蔽），會被過濾掉。
    """
    if val is None:
        return np.array([], dtype=float)
    if np.isscalar(val):
        if pd.isna(val):
            return np.array([], dtype=float)
        return np.array([float(val)], dtype=float)
    if isinstance(val, str):
        s = val.strip().strip("[]'\"").replace(",", " ")
        s = " ".join(s.split())
        if not s:
            return np.array([], dtype=float)
        return np.fromstring(s, sep=" ", dtype=float)
    arr = np.asarray(val, dtype=float).ravel()
    return arr


def clean_pixels(val, remove_zeros=True):
    """解析後過濾無效值（0 / inf / nan）"""
    arr = parse_pixel_array(val)
    if arr.size == 0:
        return arr
    if remove_zeros:
        arr = arr[arr != 0]
    arr = arr[np.isfinite(arr)]
    return arr


def safe_corr(a, b):
    """計算兩個 array 的相關係數，處理各種邊界情況"""
    a = np.asarray(a, dtype=float).ravel()
    b = np.asarray(b, dtype=float).ravel()
    m = min(a.size, b.size)
    if m < 3:
        return np.nan
    a, b = a[:m], b[:m]
    mask = np.isfinite(a) & np.isfinite(b)
    a, b = a[mask], b[mask]
    if a.size < 3 or np.std(a) == 0 or np.std(b) == 0:
        return np.nan
    return float(np.corrcoef(a, b)[0, 1])

In [ ]:
# ============================================================
# 2  計算單一像素陣列的完整統計量
#    所有統計都在「該觀測點自己的 pixel window」內計算
#    → 不會跨點汙染，不會引發 spatial autocorrelation
# ============================================================
BAND_COLS = ['blue', 'green', 'red', 'nir', 'swir16', 'swir22', 'st_b10']

def pixel_stats(arr):
    """
    給定一個 clean pixel array，回傳字典格式的統計量。
    若 array 為空，所有值回傳 NaN。
    """
    if arr.size == 0:
        return dict(
            mean=np.nan, median=np.nan, std=np.nan,
            p25=np.nan, p75=np.nan, iqr=np.nan,
            cv=np.nan, skew=np.nan, kurt=np.nan,
            n_valid=0
        )
    mn   = float(np.nanmean(arr))
    med  = float(np.nanmedian(arr))
    sd   = float(np.nanstd(arr))
    p25  = float(np.nanpercentile(arr, 25))
    p75  = float(np.nanpercentile(arr, 75))
    iqr  = p75 - p25
    cv   = sd / (mn + 1e-10)               # 變異係數（空間異質性指標）
    sk   = float(stats.skew(arr))          if arr.size >= 3 else np.nan
    ku   = float(stats.kurtosis(arr))      if arr.size >= 4 else np.nan
    return dict(
        mean=mn, median=med, std=sd,
        p25=p25, p75=p75, iqr=iqr,
        cv=cv, skew=sk, kurt=ku,
        n_valid=int(arr.size)
    )


def compute_band_stats_row(row):
    """對單一 row 的所有波段計算像素統計，展平成 Series"""
    out = {}
    for band in BAND_COLS:
        arr = clean_pixels(row.get(band))
        s   = pixel_stats(arr)
        for k, v in s.items():
            out[f'{band}__{k}'] = v
    return pd.Series(out)


def compute_all_band_stats(raw_df):
    """向量化：對整個 DataFrame 計算波段統計"""
    print('  計算波段統計特徵 (pixel stats)...')
    stats_df = raw_df.progress_apply(compute_band_stats_row, axis=1)
    return stats_df

In [ ]:
# ============================================================
# 3  光譜指數：全部用各波段的 MEDIAN 值計算
#    (median 比 mean 更抗雲陰影等 outlier)
#
#  水質導向指數（首選）
#  ─────────────────────────────────────────────────────────
#  NDWI    (McFeeters): (G - NIR) / (G + NIR)         水體識別
#  MNDWI   (Xu):        (G - SWIR16) / (G + SWIR16)   改良水體
#  NDMI:               (NIR - SWIR16) / (NIR + SWIR16) 水分含量
#  AWEInsh:            4*(G-SWIR16) - 0.25*NIR + 2.75*SWIR22  陰影不敏感
#  AWEIsh:             G + 2.5*NIR - 1.5*(SWIR16+SWIR22) - 0.25*BLUE  陰影敏感
#  WRI:                (G+R)/(NIR+SWIR16)              水體比率
#  Turbidity_proxy:    R / G                            濁度代理
#  CDOM_proxy:         B / G                            有色溶解有機物代理
#  FUI_proxy:          ln(B/G)                          Forel-Ule 顏色指數代理
#  Chl_proxy:          (R-NIR) / (R+NIR)               葉綠素 a 代理
#
#  植被/土地覆蓋指數（輔助特徵）
#  ─────────────────────────────────────────────────────────
#  NDVI, NDBI, EVI, NDTI, UrbanScore
# ============================================================

def compute_spectral_indices(stats_df, eps=1e-10):
    """
    輸入：band_stats_df（含 blue__median, green__median ... 等欄位）
    輸出：新增光譜指數欄位的 DataFrame
    """
    df = stats_df.copy()

    def s(band):  # 取 median 欄位，不存在就傳回 NaN Series
        col = f'{band}__median'
        return df[col] if col in df.columns else pd.Series(np.nan, index=df.index)

    B   = s('blue')
    G   = s('green')
    R   = s('red')
    N   = s('nir')
    S16 = s('swir16')
    S22 = s('swir22')
    LST = s('st_b10')

    # ── 水體 ──────────────────────────────────────────────────
    df['SI_NDWI']         = (G - N)          / (G + N + eps)
    df['SI_MNDWI']        = (G - S16)        / (G + S16 + eps)
    df['SI_NDMI']         = (N - S16)        / (N + S16 + eps)
    df['SI_AWEInsh']      = 4*(G - S16) - 0.25*N + 2.75*S22
    df['SI_AWEIsh']       = G + 2.5*N - 1.5*(S16 + S22) - 0.25*B
    df['SI_WRI']          = (G + R)          / (N + S16 + eps)

    # ── 水質代理 ──────────────────────────────────────────────
    df['SI_Turbidity']    = R / (G + eps)                      # 高 → 高濁度
    df['SI_CDOM']         = B / (G + eps)                      # 高 → CDOM 偏高
    df['SI_FUI']          = np.log(B / (G + eps) + 1)         # Forel-Ule 代理
    df['SI_Chl_proxy']    = (R - N) / (R + N + eps)           # 負值 → 藻類多
    df['SI_Sed_proxy']    = (R - B) / (R + B + eps)           # 懸浮沉積物代理
    df['SI_BR_ratio']     = B / (R + eps)                      # 藍紅比
    df['SI_GR_ratio']     = G / (R + eps)                      # 綠紅比（水色指標）

    # ── 植被/土地覆蓋（控制陸地混入）────────────────────────
    df['SI_NDVI']         = (N - R)   / (N + R + eps)
    df['SI_NDTI']         = (R - G)   / (R + G + eps)
    df['SI_NDBI']         = (S16 - N) / (S16 + N + eps)
    df['SI_EVI']          = 2.5*(N - R) / (N + 6*R - 7.5*B + 1 + eps)
    df['SI_UrbanScore']   = df['SI_NDBI'] - df['SI_NDVI']

    # ── 熱紅外衍生 ────────────────────────────────────────────
    # LST 用 st_b10 median，保留為獨立欄位即可（已在 band stats 中）
    # 這裡加 LST × NDVI 交互項（植被蒸散降溫效應）
    df['SI_LST_NDVI']     = LST * df['SI_NDVI']
    df['SI_LST_MNDWI']    = LST * df['SI_MNDWI']

    print(f'  光譜指數：{[c for c in df.columns if c.startswith("SI_")]}')
    return df

In [ ]:
# ============================================================
# 4  Pixel-level 特徵
#    在每個觀測點的 pixel window 內計算：
#    - 水體 / 植被 / 裸地 / 城市 像素比例
#    - 像素級指數與 LST 的相關性
#    - 熱點比例
#    這些特徵完全在「單點 pixel window」內計算，不跨點
# ============================================================

def compute_pixel_features_row(
    row,
    eps=1e-10,
    ndvi_thr=0.3,
    ndbi_thr=0.0,
    mndwi_thr=0.0,
    ndwi_thr=0.0,
    ndti_thr=0.2,
    hot_q=0.75
):
    """
    從 pixel array 計算像素層級特徵（比例 / 相關 / 熱點）
    """
    B   = clean_pixels(row.get('blue'))
    G   = clean_pixels(row.get('green'))
    R   = clean_pixels(row.get('red'))
    N   = clean_pixels(row.get('nir'))
    S16 = clean_pixels(row.get('swir16'))
    S22 = clean_pixels(row.get('swir22'))
    LST = clean_pixels(row.get('st_b10'))

    def align(*arrs):
        m = min((a.size for a in arrs), default=0)
        return [a[:m] for a in arrs] if m > 0 else [np.array([]) for _ in arrs]

    out = {}

    # ── 像素級指數 ──────────────────────────────────────────
    N_a, R_a    = align(N, R)
    G_a, S16_a  = align(G, S16)
    G_b, N_b    = align(G, N)
    S16_c, N_c  = align(S16, N)
    R_d, G_d    = align(R, G)

    ndvi  = (N_a - R_a)   / (N_a + R_a + eps)   if N_a.size else np.array([])
    mndwi = (G_a - S16_a) / (G_a + S16_a + eps) if G_a.size else np.array([])
    ndwi  = (G_b - N_b)   / (G_b + N_b + eps)   if G_b.size else np.array([])
    ndbi  = (S16_c - N_c) / (S16_c + N_c + eps) if S16_c.size else np.array([])
    ndti  = (R_d - G_d)   / (R_d + G_d + eps)   if R_d.size else np.array([])

    # ── 土地覆蓋比例 ────────────────────────────────────────
    out['px_water_MNDWI']  = float(np.nanmean(mndwi > mndwi_thr)) if mndwi.size else np.nan
    out['px_water_NDWI']   = float(np.nanmean(ndwi  > ndwi_thr))  if ndwi.size else np.nan
    out['px_veg']          = float(np.nanmean(ndvi  > ndvi_thr))  if ndvi.size else np.nan
    out['px_bare']         = float(np.nanmean(ndti  > ndti_thr))  if ndti.size else np.nan

    # urban: NDBI > 0 and NDVI < 0.2
    if ndbi.size > 0 and ndvi.size > 0:
        ndbi_a, ndvi_a = align(ndbi, ndvi)
        out['px_urban'] = float(np.nanmean((ndbi_a > ndbi_thr) & (ndvi_a < 0.2)))
    else:
        out['px_urban'] = np.nan

    # ── 熱點比例 ────────────────────────────────────────────
    if LST.size > 0:
        q75 = np.nanquantile(LST, hot_q)
        out['px_hot_ratio'] = float(np.nanmean(LST > q75))
        out['px_lst_cv']    = float(np.nanstd(LST) / (np.nanmean(LST) + 1e-10))
    else:
        out['px_hot_ratio'] = np.nan
        out['px_lst_cv']    = np.nan

    # ── 指數與 LST 的相關係數 ─────────────────────────────
    out['px_corr_NDVI_LST']  = safe_corr(ndvi,  LST)
    out['px_corr_NDBI_LST']  = safe_corr(ndbi,  LST)
    out['px_corr_MNDWI_LST'] = safe_corr(mndwi, LST)
    out['px_corr_NDWI_LST']  = safe_corr(ndwi,  LST)

    # ── 波段異質性：各波段像素的 CV ──────────────────────────
    # CV = std / mean → 衡量 pixel window 內的空間變異程度
    # 高 CV → 表示採樣點附近地物混合複雜（邊緣效應）
    for band, arr in [('B', B), ('G', G), ('R', R), ('N', N),
                       ('S16', S16), ('S22', S22)]:
        if arr.size > 1:
            out[f'px_cv_{band}'] = float(np.nanstd(arr) / (np.nanmean(arr) + 1e-10))
        else:
            out[f'px_cv_{band}'] = np.nan

    # ── 有效像素數（proxy for 雲遮蔽程度）──────────────────
    out['px_n_valid'] = int(N.size) if N.size else 0

    return pd.Series(out)


def compute_all_pixel_features(raw_df):
    print('  計算像素層級特徵 (ratios / corr / heterogeneity)...')
    return raw_df.progress_apply(compute_pixel_features_row, axis=1)

In [ ]:
# ============================================================
# 5  時間特徵
#    所有特徵只從 Sample Date 推導，不使用 Lat/Lon，
#    不會引入 spatial autocorrelation
# ============================================================

def compute_temporal_features(df):
    """
    從 Sample Date 提取時間特徵。
    """
    dt = pd.to_datetime(df['Sample Date'], dayfirst=True, errors='coerce')

    month  = dt.dt.month
    doy    = dt.dt.dayofyear   # Day of Year
    year   = dt.dt.year

    # 循環編碼：把月份/DOY 映射到單位圓上
    # → 避免 1 月和 12 月被模型認為距離很遠
    month_sin  = np.sin(2 * np.pi * month / 12)
    month_cos  = np.cos(2 * np.pi * month / 12)
    doy_sin    = np.sin(2 * np.pi * doy   / 365)
    doy_cos    = np.cos(2 * np.pi * doy   / 365)

    # 南半球季節（12-2月為夏季）
    def get_season(m):
        if m in [12, 1, 2]:  return 0   # Summer
        if m in [3, 4, 5]:   return 1   # Autumn
        if m in [6, 7, 8]:   return 2   # Winter
        return 3                          # Spring

    season = month.apply(get_season)  # 0/1/2/3（整數，樹模型不需要 one-hot）

    # 南非主要「乾季(5-10) / 濕季(11-4)」二元特徵
    is_wet_season = (month.isin([11, 12, 1, 2, 3, 4])).astype(int)

    return pd.DataFrame({
        'T_year':       year.values,
        'T_month':      month.values,
        'T_doy':        doy.values,
        'T_season':     season.values,
        'T_is_wet':     is_wet_season.values,
        'T_month_sin':  month_sin.values,
        'T_month_cos':  month_cos.values,
        'T_doy_sin':    doy_sin.values,
        'T_doy_cos':    doy_cos.values,
    }, index=df.index)

In [ ]:
# ============================================================
# 6  主 Pipeline
#
# 輸入：landsat raw parquet (含 Latitude / Longitude /
#        Sample Date / blue / green / red / nir / swir16 / swir22 / st_b10)
# 輸出：feature DataFrame（不含 Lat/Lon 以避免模型學習地理位置）
#
# ⚠️  Lat/Lon 保留為 join key，但不進入 X matrix
#     （若納入 Lat/Lon 等地理座標，模型會學到
#       「在哪裡」而非「看到什麼」，等同引入 spatial autocorrelation）
# ============================================================

from tqdm import tqdm
tqdm.pandas()

def process_landsat_pipeline(raw_df, label='dataset'):
    """
    完整 Feature Engineering Pipeline。

    Parameters
    ----------
    raw_df : pd.DataFrame
        含 Latitude / Longitude / Sample Date / 7 波段 的 raw parquet 資料
    label : str
        資料集名稱（用於 print 訊息）

    Returns
    -------
    features_df : pd.DataFrame
        含 join key (Latitude / Longitude / Sample Date) +
        所有 feature 欄位的 DataFrame
    """
    print(f'\n====== Processing [{label}] | {len(raw_df)} rows ======')

    # ── Step 1：像素統計（mean / median / std / percentile / CV / skew / kurt）
    print('Step 1/4: Band pixel statistics')
    band_stats = compute_all_band_stats(raw_df)

    # ── Step 2：光譜指數（用各波段 median 計算）
    print('Step 2/4: Spectral indices')
    with_indices = compute_spectral_indices(band_stats)

    # ── Step 3：像素層級特徵（比例 / 相關 / 異質性）
    print('Step 3/4: Pixel-level features (ratios, heterogeneity, correlations)')
    pixel_feats = compute_all_pixel_features(raw_df)

    # ── Step 4：時間特徵
    print('Step 4/4: Temporal features')
    temporal_feats = compute_temporal_features(raw_df)

    # ── 合併 ──────────────────────────────────────────────────
    base = raw_df[['Latitude', 'Longitude', 'Sample Date']].reset_index(drop=True)
    features_df = pd.concat([
        base,
        with_indices.reset_index(drop=True),
        pixel_feats.reset_index(drop=True),
        temporal_feats.reset_index(drop=True),
    ], axis=1)

    # 去除重複欄位（n_valid 可能重複）
    features_df = features_df.loc[:, ~features_df.columns.duplicated()]

    print(f'\n完成！輸出形狀: {features_df.shape}')
    print(f'特徵數（不含 key）: {features_df.shape[1] - 3}')
    print(f'NaN 比例: {features_df.iloc[:, 3:].isnull().mean().mean():.3f}')
    return features_df

In [ ]:
# ── 讀取 raw data ──────────────────────────────────────────
landsat_train_raw = pd.read_parquet('landsat_raw_all_v3.parquet')
landsat_val_raw   = pd.read_parquet('landsat_validation_raw_all_v3.parquet')

print(f'Train raw: {landsat_train_raw.shape}')
print(f'Val raw:   {landsat_val_raw.shape}')

In [ ]:
# ── 執行 Pipeline ──────────────────────────────────────────
# Train 和 Validation 各自獨立執行，不共用任何統計量
train_features = process_landsat_pipeline(landsat_train_raw, label='TRAIN')
val_features   = process_landsat_pipeline(landsat_val_raw,   label='VALIDATION')

In [ ]:
# ── 儲存 ───────────────────────────────────────────────────
train_features.to_csv('/tmp/landsat_features_training_v2.csv', index=False)
val_features.to_csv('/tmp/landsat_features_validation_v2.csv', index=False)

print('Train features saved → landsat_features_training_v2.csv')
print('Val   features saved → landsat_features_validation_v2.csv')
train_features.head(3)

In [ ]:
import snowflake
from snowflake.snowpark.context import get_active_session
session = get_active_session()


session.sql("""
    PUT file:///tmp/landsat_features_training_v2.csv
    'snow://workspace/USER$.PUBLIC."EY-AI-and-Data-Challenge"/versions/live/'
    AUTO_COMPRESS=FALSE
    OVERWRITE=TRUE
""").collect()

print("File saved! Refresh the browser to see the files in the sidebar")

session.sql("""
    PUT file:///tmp/landsat_features_validation_v2.csv
    'snow://workspace/USER$.PUBLIC."EY-AI-and-Data-Challenge"/versions/live/'
    AUTO_COMPRESS=FALSE
    OVERWRITE=TRUE
""").collect()

print("File saved! Refresh the browser to see the files in the sidebar")